# N161 · JavaScript基础、闭包与对象

**目标：** 理解JS与Python语义差异并保留闭包状态。

**先修：** N004, N009, N010。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** let/const；类型与===；undefined/null；数组对象；闭包；this/原型导览。

## 从问题到算法

闭包捕获的是词法环境中的绑定，不是调用时复制的常数。创建计数器只执行一次初始化，随后每次调用更新同一个私有 `current`。重新创建计数器则产生独立环境。

普通计数器返回旧值后递增；带复位对象的 `increment` 返回新值，`reset` 恢复构造时的初值。这个差别是返回值契约，不能用看似相近的 `++` 写法混过去。

以下代码单元格保存原生 JS 源码并用 Node.js 执行；Python 只负责组织 Notebook 和显示结果。

## 最小实现

**本章接口：** `createHelloWorld()`、`createCounter(n)`、`createCounterWithReset(init)`

In [1]:
JS_SOURCE = r''''use strict';
function createHelloWorld() {
  return function (...args) { return 'Hello World'; };
}
function createCounter(n) {
  let current = n;
  return function () { return current++; };
}
function createCounterWithReset(init) {
  let current = init;
  return {
    increment() { return ++current; },
    decrement() { return --current; },
    reset() { current = init; return current; }
  };
}
'''

import json
import subprocess

def run_js(extra):
    result = subprocess.run(['node', '-e', JS_SOURCE + '\n' + extra],
                            text=True, capture_output=True, check=True, timeout=20)
    return result.stdout

## 正确性、前提与复杂度

每次工厂调用创建独立 `current`，对象方法共享同一词法环境。对调用次数归纳可知普通计数器第 k 次返回 n+k−1；复位将状态转回初态。

**代价：** 单次构造、调用均 O(1)；每个计数器保存 O(1) 状态。不涉及 JavaScript 数值超出安全整数范围后的精确大整数计数。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

from IPython.display import Code
display(Code(JS_SOURCE,language='javascript'))
trace = json.loads(run_js("const a=createCounter(5), b=createCounter(-1); console.log(JSON.stringify([['a',a()],['a',a()],['b',b()],['a',a()]]));"))
show_table(['closure','returned value'],trace)
print(run_js('console.log(process.version);').strip())

'use strict';
function createHelloWorld() {
  return function (...args) { return 'Hello World'; };
}
function createCounter(n) {
  let current = n;
  return function () { return current++; };
}
function createCounterWithReset(init) {
  let current = init;
  return {
    increment() { return ++current; },
    decrement() { return --current; },
    reset() { current = init; return current; }
  };
}

closure,returned value
a,5
a,6
b,-1
a,7


v22.16.0


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
print(run_js("""const assert=require('node:assert/strict');
const hello=createHelloWorld();
assert.equal(hello(),'Hello World'); assert.equal(hello(1,null,{}),'Hello World');
const a=createCounter(3),b=createCounter(3);
assert.deepEqual([a(),a(),b(),a(),b()],[3,4,3,5,4]);
const c=createCounterWithReset(5);
assert.deepEqual([c.increment(),c.reset(),c.decrement(),c.reset()],[6,5,4,5]);
console.log('Node assertions passed');"""))
print('N161: 所有本章断言通过')

Node assertions passed

N161: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 比较重新创建计数器和复用计数器。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释箭头函数的this差异。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 2667. Create Hello World Function（canonical）：[官方入口](https://leetcode.com/problems/create-hello-world-function/)
- 2620. Counter（canonical）
- 2665. Counter II（canonical）
- 2695. Array Wrapper（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。